# Latent Depth Metrics Analysis

Analyze fixed-THINK latent-depth outputs with per-step attention mass and hidden-state norm metrics.

In [ ]:
%matplotlib inline
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'lvar').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change this to the root containing fixed_think_steps_0 ... fixed_think_steps_10.
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'inference' / 'fixed_think_sweep' / 'm3cot' / 'test'
JSONL_GLOB = '**/*.jsonl'
EXPECTED_DEPTHS = list(range(11))
print('RUN_ROOT =', RUN_ROOT)

## Load Fixed-Depth Outputs

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def infer_depth(row, path):
    if row.get('num_think_steps') is not None:
        return int(row['num_think_steps'])
    matches = re.findall(r'(?:fixed[_-]?think[_-]?steps?|latent[_-]?steps?)[_-]?(\d+)', str(path))
    if matches:
        return int(matches[-1])
    return np.nan

rows = []
for path in sorted(RUN_ROOT.glob(JSONL_GLOB)) if RUN_ROOT.exists() else []:
    if path.name.endswith('_summary.json') or path.name.endswith('_entropy_tracking.json'):
        continue
    for row in read_jsonl(path):
        row = dict(row)
        row['latent_depth'] = infer_depth(row, path)
        row['source_path'] = str(path)
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Loaded {len(df):,} rows')
display(df[['example_id', 'latent_depth', 'correct']].head() if not df.empty else df)

## Flatten Per-Step Metrics

In [ ]:
attention_rows = []
hidden_rows = []
for row in rows:
    base = {
        'example_id': row.get('example_id'),
        'latent_depth': row.get('latent_depth'),
        'correct': bool(row.get('correct')),
    }
    for metric in row.get('latent_step_attention_summary') or []:
        attention_rows.append({**base, **metric})
    for metric in row.get('latent_step_hidden_metrics') or []:
        hidden_rows.append({**base, **metric})

attn_df = pd.DataFrame(attention_rows)
hidden_df = pd.DataFrame(hidden_rows)
print(f'Attention metric rows: {len(attn_df):,}')
print(f'Hidden metric rows: {len(hidden_df):,}')
display(attn_df.head() if not attn_df.empty else attn_df)
display(hidden_df.head() if not hidden_df.empty else hidden_df)

## Bare Minimum Plots

In [ ]:
if attn_df.empty:
    print('No latent_step_attention_summary fields found. Re-run inference with --track-latent-depth-metrics.')
else:
    summary = attn_df.groupby('step_idx', as_index=False).agg(
        visual_attention_mass=('image_mass', 'mean'),
        prompt_attention_mass=('prompt_mass', 'mean'),
        latent_attention_mass=('latent_mass', 'mean'),
    )
    ax = summary.plot(x='step_idx', y='visual_attention_mass', marker='o', figsize=(8, 4))
    summary.plot(x='step_idx', y='prompt_attention_mass', marker='o', ax=ax)
    summary.plot(x='step_idx', y='latent_attention_mass', marker='o', ax=ax)
    ax.set_xlabel('Latent step')
    ax.set_ylabel('Attention mass')
    ax.set_title('Attention mass by latent step')
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
if hidden_df.empty:
    print('No latent_step_hidden_metrics fields found. Re-run inference with --track-latent-depth-metrics.')
else:
    summary = hidden_df.groupby('step_idx', as_index=False).agg(
        hidden_norm=('hidden_norm', 'mean'),
        hidden_norm_delta=('hidden_norm_delta', 'mean'),
        hidden_delta_norm=('hidden_delta_norm', 'mean'),
    )
    ax = summary.plot(x='step_idx', y='hidden_norm', marker='o', figsize=(8, 4))
    ax.set_xlabel('Latent step')
    ax.set_ylabel('Hidden-state norm')
    ax.set_title('Latent hidden-state norm by step')
    ax.grid(True, alpha=0.3)
    plt.show()

    ax = summary.plot(x='step_idx', y=['hidden_norm_delta', 'hidden_delta_norm'], marker='o', figsize=(8, 4))
    ax.set_xlabel('Latent step')
    ax.set_ylabel('Change')
    ax.set_title('Latent representation change by step')
    ax.grid(True, alpha=0.3)
    plt.show()

## Correctness And Oracle Depth

In [ ]:
if df.empty:
    print('No prediction rows loaded.')
else:
    acc = df.groupby('latent_depth', as_index=False).agg(accuracy=('correct', 'mean'), total=('correct', 'size'))
    acc['accuracy_pct'] = 100 * acc['accuracy']
    display(acc)
    ax = acc.plot(x='latent_depth', y='accuracy_pct', marker='o', figsize=(8, 4), legend=False)
    ax.set_xlabel('Latent depth')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by fixed latent depth')
    ax.grid(True, alpha=0.3)
    plt.show()

    oracle_rows = []
    for example_id, group in df.groupby('example_id'):
        correct_steps = sorted(group.loc[group['correct'], 'latent_depth'].dropna().astype(int).tolist())
        oracle_rows.append({
            'example_id': example_id,
            'oracle_correct': bool(correct_steps),
            'correct_depths': correct_steps,
            'earliest_correct_depth': correct_steps[0] if correct_steps else np.nan,
            'num_correct_depths': len(correct_steps),
        })
    oracle_df = pd.DataFrame(oracle_rows)
    display(oracle_df.head())
    print('Oracle accuracy:', oracle_df['oracle_correct'].mean() if not oracle_df.empty else np.nan)
    display(oracle_df['earliest_correct_depth'].value_counts(dropna=False).sort_index())

## Per-Layer Attention Heatmap

In [ ]:
layer_rows = []
for row in rows:
    for step in row.get('latent_step_attention') or []:
        for layer in step.get('per_layer') or []:
            layer_rows.append({
                'example_id': row.get('example_id'),
                'step_idx': step.get('step_idx'),
                'layer_idx': layer.get('layer_idx'),
                'image_mass': (layer.get('image') or {}).get('mean'),
                'prompt_mass': (layer.get('prompt') or {}).get('mean'),
                'latent_mass': (layer.get('latent') or {}).get('mean'),
            })
layer_df = pd.DataFrame(layer_rows)
if layer_df.empty:
    print('No per-layer attention rows available.')
else:
    pivot = layer_df.pivot_table(index='layer_idx', columns='step_idx', values='image_mass', aggfunc='mean')
    plt.figure(figsize=(10, 5))
    plt.imshow(pivot.values, aspect='auto', interpolation='nearest')
    plt.colorbar(label='Visual attention mass')
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel('Latent step')
    plt.ylabel('Layer')
    plt.title('Per-layer visual attention mass')
    plt.show()